# Home Energy Helper Agent

This notebook starts with a basic RAG baseline, then adds a tool-using agent that can compare two bill periods and search DOE Energy Saver guidance.

In [39]:
import json
import re
from datetime import datetime, timedelta, timezone
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from minsearch import Index
from pypdf import PdfReader

load_dotenv()
openai_client = OpenAI()


## 1. Fetch and Chunk Energy.gov Guidance

The knowledge base is fetched from a small, curated set of U.S. Department of Energy pages. Extracted text is chunked with source metadata preserved, then cached locally in the git-ignored `data/energygov_cache.json` file. Set `force_refresh=True` to fetch the sources again.


In [40]:
ENERGY_GOV_SOURCES = [
    {
        "title": "Why Energy Efficiency Matters",
        "topic": "home energy efficiency, energy audits, HVAC, appliances, lighting",
        "url": "https://www.energy.gov/energysaver/why-energy-efficiency-matters",
    },
    {
        "title": "Heat Pump Systems",
        "topic": "heating, cooling, heat pumps, electric resistance heating",
        "url": "https://www.energy.gov/energysaver/heat-pump-systems",
    },
    {
        "title": "Consumer Guide to Air Sealing the Home",
        "topic": "weatherization, air leaks, drafts, caulking, weatherstripping",
        "url": "https://www.energy.gov/sites/default/files/2021-08/ES-Air%20Sealing_080421.pdf",
    },
    {
        "title": "Consumer Water Heaters",
        "topic": "water heating, heat pump water heaters, efficiency",
        "url": "https://www.energy.gov/cmei/buildings/consumer-water-heaters",
    },
    {
        "title": "Renters",
        "topic": "low-cost energy savings, renters, thermostats, drafts",
        "url": "https://www.energy.gov/save/renters",
    },
]

CACHE_PATH = Path("data/energygov_cache.json")
CACHE_MAX_AGE = timedelta(days=7)


def extract_source_text(source, response_bytes):
    """Extract readable text from an Energy.gov HTML page or PDF."""
    if source["url"].lower().split("?")[0].endswith(".pdf"):
        reader = PdfReader(BytesIO(response_bytes))
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    soup = BeautifulSoup(response_bytes, "html.parser")
    for element in soup.select("script, style, nav, footer, header, noscript"):
        element.decompose()
    main = soup.select_one("main") or soup.select_one("article") or soup.body or soup
    return main.get_text(" ", strip=True)


def fetch_energy_gov_sources():
    """Fetch the curated DOE pages, tolerating individual source failures."""
    documents = []
    failures = []
    for source in ENERGY_GOV_SOURCES:
        try:
            request = Request(
                source["url"],
                headers={"User-Agent": "HomeEnergyHelper/0.1 (educational project)"},
            )
            with urlopen(request, timeout=25) as response:
                text = extract_source_text(source, response.read())
            if len(text.split()) < 50:
                raise ValueError("the extracted page contained too little readable text")
            documents.append({**source, "source_name": "U.S. Department of Energy", "text": text})
            print(f"Fetched: {source['title']}")
        except Exception as exc:
            failures.append(f"{source['title']}: {exc}")
            print(f"Could not fetch {source['title']}: {exc}")
    return documents, failures


def chunk_documents(documents, chunk_words=450, overlap_words=75):
    """Create overlapping word chunks while retaining each DOE citation."""
    chunks = []
    for document in documents:
        words = document["text"].split()
        step = chunk_words - overlap_words
        for start in range(0, len(words), step):
            content = " ".join(words[start : start + chunk_words]).strip()
            if not content:
                continue
            chunks.append(
                {
                    "title": document["title"],
                    "topic": document["topic"],
                    "content": content,
                    "source_name": document["source_name"],
                    "source_url": document["url"],
                }
            )
    return chunks


def load_energy_chunks(force_refresh=False):
    """Load a fresh local cache or fetch the Energy.gov sources and refresh it."""
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    cached = None
    if CACHE_PATH.exists():
        try:
            cached = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
            fetched_at = datetime.fromisoformat(cached["fetched_at"])
            if not force_refresh and datetime.now(timezone.utc) - fetched_at < CACHE_MAX_AGE:
                print(f"Using cached Energy.gov text fetched {fetched_at.date()}.")
                return cached["chunks"]
        except (OSError, ValueError, KeyError, json.JSONDecodeError):
            cached = None

    documents, failures = fetch_energy_gov_sources()
    if documents:
        chunks = chunk_documents(documents)
        cache = {
            "fetched_at": datetime.now(timezone.utc).isoformat(),
            "chunks": chunks,
        }
        CACHE_PATH.write_text(json.dumps(cache, indent=2), encoding="utf-8")
        if failures:
            print(f"Fetched {len(documents)} source(s); {len(failures)} source(s) failed.")
        print(f"Created {len(chunks)} chunks in {CACHE_PATH}.")
        return chunks

    if cached and cached.get("chunks"):
        print("All refreshes failed; using the existing cached Energy.gov content.")
        return cached["chunks"]
    raise RuntimeError(
        "No Energy.gov pages could be fetched and no cache exists. Check your internet "
        "connection, then rerun this cell."
    )


energy_chunks = load_energy_chunks()
print(f"Knowledge base ready: {len(energy_chunks)} chunks from Energy.gov.")


Could not fetch Why Energy Efficiency Matters: HTTP Error 404: Not Found
Could not fetch Heat Pump Systems: HTTP Error 404: Not Found
Fetched: Consumer Guide to Air Sealing the Home
Fetched: Consumer Water Heaters
Fetched: Renters
Fetched 3 source(s); 2 source(s) failed.
Created 15 chunks in data\energygov_cache.json.
Knowledge base ready: 15 chunks from Energy.gov.


## 2. Build the Index

Index the Energy.gov chunks by their title, topic, and passage text; retain the source URL and name as metadata for citations.


In [41]:
index = Index(
    text_fields=["title", "topic", "content"],
    keyword_fields=["source_name", "source_url"],
)
index.fit(energy_chunks)


## 3. Basic RAG Functions

These functions preserve the starter's `search`, `build_prompt`, `llm`, and `rag` structure while using fetched Energy.gov chunks as the knowledge base.


In [42]:
def search(query):
    return index.search(query, num_results=5)

In [43]:
rag_instructions = """
You are a careful Home Energy Helper. Answer only questions about household energy use, electricity bills, and practical energy-saving guidance. For guidance, use only the supplied Energy.gov excerpts; do not answer from general model knowledge. Cite every factual recommendation with a Markdown link using the exact title and source URL from an excerpt. If the excerpts do not support an answer, say that the knowledge base does not provide enough information and ask one focused follow-up question. Do not guess the cause of a bill change: usage records show what changed, not why. Distinguish calculations from explanations, and never invent source links or savings claims.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)
    return f"""
<QUESTION>
{query}
</QUESTION>

<ENERGY_GOV_CONTEXT>
{search_result_json}
</ENERGY_GOV_CONTEXT>
""".strip()


In [44]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [45]:
def rag(query):
    search_results = search(query)
    if not search_results:
        return "I couldn't find relevant Energy.gov guidance for that question. Could you rephrase it around a home energy topic?"
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions=rag_instructions)


## 4. PydanticAI Tools and Agent

PydanticAI manages tool calls for bill comparisons. The application also retrieves DOE passages before the model runs, so answers are grounded even when the model would otherwise skip the search tool. Calls to both focused tools are printed for visibility.


The agent uses the same `search`, `build_prompt`, `llm`, and `rag` baseline functions, plus focused tools for guidance retrieval and bill comparisons. A small deterministic scope check rejects unrelated requests before they reach the model.


In [46]:
from math import isfinite


class BillPeriod(BaseModel):
    """Bill details for one month; provide billing_days when known."""

    month: str = Field(description="The bill-period label exactly as the user gave it; do not infer a year.")
    kwh: float = Field(ge=0, description="Electricity use in kilowatt-hours.")
    bill_amount: float = Field(ge=0, description="Total bill amount in the currency stated by the user.")
    billing_days: int | None = Field(default=None, ge=1, description="Number of billing days, or null if unknown.")


class HomeEnergyTools:
    """Small, focused tools registered with the PydanticAI agent."""

    def search_energy_guidance(self, question: str) -> list[dict]:
        """Search fetched Energy.gov chunks and return relevant excerpts with source links."""
        print(f"TOOL CALL: search_energy_guidance({question!r})")
        results = search(question)
        print(f"TOOL RESULT: search_energy_guidance returned {len(results)} records")
        return [
            {
                "title": result.get("title"),
                "topic": result.get("topic"),
                "content": result.get("content"),
                "source_name": result.get("source_name"),
                "source_url": result.get("source_url"),
            }
            for result in results
        ]

    @staticmethod
    def _validate_period(period: dict, label: str) -> None:
        if not isinstance(period.get("month"), str) or not period["month"].strip():
            raise ValueError(f"{label}.month must be a non-empty string")

        for field in ("kwh", "bill_amount"):
            value = period.get(field)
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise ValueError(f"{label}.{field} must be a number")
            if not isfinite(value) or value < 0:
                raise ValueError(f"{label}.{field} must be finite and non-negative")

        days = period.get("billing_days")
        if days is not None and (isinstance(days, bool) or not isinstance(days, int) or days < 1):
            raise ValueError(f"{label}.billing_days must be a positive integer or null")

    def compare_monthly_bills(
        self,
        previous_period: BillPeriod,
        current_period: BillPeriod,
    ) -> dict:
        """Compare two user-provided bill periods; report totals and daily averages when possible."""
        previous_period = previous_period.model_dump()
        current_period = current_period.model_dump()
        self._validate_period(previous_period, "previous_period")
        self._validate_period(current_period, "current_period")
        print(
            "TOOL CALL: compare_monthly_bills "
            f"({previous_period['month']} -> {current_period['month']})"
        )

        def change(current, previous):
            absolute = current - previous
            percent = None if previous == 0 else (absolute / previous) * 100
            return round(absolute, 2), None if percent is None else round(percent, 1)

        kwh_change, kwh_change_percent = change(current_period["kwh"], previous_period["kwh"])
        bill_change, bill_change_percent = change(current_period["bill_amount"], previous_period["bill_amount"])

        previous_days = previous_period["billing_days"]
        current_days = current_period["billing_days"]
        if previous_days is None or current_days is None:
            previous_kwh_per_day = current_kwh_per_day = daily_kwh_change = None
            previous_bill_per_day = current_bill_per_day = daily_bill_change = None
        else:
            previous_kwh_per_day = previous_period["kwh"] / previous_days
            current_kwh_per_day = current_period["kwh"] / current_days
            previous_bill_per_day = previous_period["bill_amount"] / previous_days
            current_bill_per_day = current_period["bill_amount"] / current_days
            daily_kwh_change = current_kwh_per_day - previous_kwh_per_day
            daily_bill_change = current_bill_per_day - previous_bill_per_day
            previous_kwh_per_day = round(previous_kwh_per_day, 2)
            current_kwh_per_day = round(current_kwh_per_day, 2)
            previous_bill_per_day = round(previous_bill_per_day, 2)
            current_bill_per_day = round(current_bill_per_day, 2)
            daily_kwh_change = round(daily_kwh_change, 2)
            daily_bill_change = round(daily_bill_change, 2)

        result = {
            "previous_month": previous_period["month"],
            "current_month": current_period["month"],
            "usage_change_kwh": kwh_change,
            "usage_change_percent": kwh_change_percent,
            "bill_change_amount": bill_change,
            "bill_change_percent": bill_change_percent,
            "previous_kwh_per_day": previous_kwh_per_day,
            "current_kwh_per_day": current_kwh_per_day,
            "daily_usage_change_kwh": daily_kwh_change,
            "previous_bill_per_day": previous_bill_per_day,
            "current_bill_per_day": current_bill_per_day,
            "daily_bill_change": daily_bill_change,
            "note": "This comparison shows what changed; it does not identify the cause. A null percent means the previous value was zero. Daily averages are null when either billing period length is missing.",
        }
        print("TOOL RESULT: compare_monthly_bills completed")
        return result


home_energy_tools = HomeEnergyTools()

In [47]:
agent_instructions = """
You are the Home Energy Helper. Help users understand household electricity bills and find practical ways to save energy.

Scope and grounding:
- Answer only about home energy use, electricity bills, or energy-saving guidance. If the user's request is outside this scope, politely state the scope and do not answer the unrelated question.
- The application supplies relevant Energy.gov passages with each request. Treat them as the only source of factual energy guidance. Never fill gaps using general model knowledge.
- Cite every guidance-based claim using a Markdown link with the exact source title and URL supplied in the passages. Never invent a URL, statistic, or savings estimate.
- If the supplied passages do not support the requested guidance, say so and ask one focused follow-up question.

Tool use and bill comparisons:
- For month-over-month comparisons, call compare_monthly_bills only when the user supplied both periods' kWh and bill amounts. Preserve the values and month labels; pass null for unknown billing days. Ask only for required missing values.
- State comparison results as observations, never as causes. The calculation cannot prove that weather, heating, billing days, rates, or an appliance caused the change.
- The application already searched the Energy.gov corpus for this turn. Call search_energy_guidance only if a needed topic is absent from the supplied excerpts.
- If the user asks why a bill changed and what to check, report any comparison first, then give up to three checks supported by the supplied sources, and finish with one targeted question about the most useful missing detail.
- This version has no live weather, rate, appliance-use, or rebate lookup. Do not claim which heating option is cheapest or calculate appliance payback.
- Keep the answer concise and practical.
""".strip()

home_energy_agent = Agent(
    "openai:gpt-4o-mini",
    name="home_energy_helper",
    instructions=agent_instructions,
    tools=[
        home_energy_tools.search_energy_guidance,
        home_energy_tools.compare_monthly_bills,
    ],
)


In [48]:
SCOPE_TERMS = {
    "energy", "electricity", "electric", "power", "kwh", "bill", "bills", "utility",
    "usage", "consumption", "heating", "heat", "heater", "cooling", "hvac",
    "thermostat", "appliance", "appliances", "water", "insulation", "draft", "drafts",
    "weatherization", "weatherstripping", "lighting", "led", "refrigerator", "fridge",
    "dryer", "washer", "home", "house", "household", "apartment", "energy-saving",
    "efficient", "efficiency", "kilowatt", "kilowatts", "solar", "conditioner", "ac",
}


def is_home_energy_question(question: str) -> bool:
    """Conservatively route questions that mention home-energy concepts."""
    words = set(re.findall(r"[a-z]+", question.lower()))
    return bool(words & SCOPE_TERMS)


async def run_agent(user_question: str) -> str:
    """Reject unrelated requests, retrieve DOE evidence, then run the PydanticAI agent."""
    if not is_home_energy_question(user_question):
        return (
            "I’m the Home Energy Helper and can answer questions about household energy use, "
            "electricity bills, and ways to save energy. Is your question about one of those?"
        )

    # Retrieve before the model runs so it cannot answer a guidance question from memory alone.
    guidance = home_energy_tools.search_energy_guidance(user_question)
    has_bill_data = bool(re.search(r"\b(kwh|bill|electricity|usage)\b", user_question, re.I))
    asks_for_guidance = bool(
        re.search(
            r"\b(why|cause|check|reduce|save|lower|efficient|guidance|advice|tip|tips|heat|heating|cooling|thermostat|appliance)\b",
            user_question,
            re.I,
        )
    )
    if asks_for_guidance and not guidance:
        return (
            "I couldn't find relevant Energy.gov passages to support guidance for that question. "
            "Could you share a more specific home energy topic, such as heating, water heating, or air leaks?"
        )
    if not guidance and not has_bill_data:
        return "I couldn't find relevant Energy.gov guidance for that question. Could you rephrase it around a home energy topic?"

    grounded_question = f"""
USER QUESTION:
{user_question}

RETRIEVED ENERGY.GOV PASSAGES (use only these for factual energy guidance):
{json.dumps(guidance, indent=2)}
""".strip()
    result = await home_energy_agent.run(grounded_question)
    return result.output


## 5. Try the Agent
 
 The examples show a bill comparison, a guidance question, and an unrelated question that the scope check should decline.


In [49]:
answer = await run_agent(
    "My electricity use went from 620 kWh and $124 over 31 days in January to 790 kWh and $158 over 28 days in February. Why is my bill higher, and what should I check?"
)
print(answer)


TOOL CALL: search_energy_guidance('My electricity use went from 620 kWh and $124 over 31 days in January to 790 kWh and $158 over 28 days in February. Why is my bill higher, and what should I check?')
TOOL RESULT: search_energy_guidance returned 5 records
TOOL CALL: compare_monthly_bills (January -> February)
TOOL RESULT: compare_monthly_bills completed
Your electricity bill increased significantly from January to February. Here's a summary of the comparison:

### Comparison:
- **January**: 620 kWh, $124 over 31 days (about 20 kWh/day, $4.00/day)
- **February**: 790 kWh, $158 over 28 days (about 28.21 kWh/day, $5.64/day)
- **Increased Usage**: 170 kWh (approximately 27.4%)
- **Increased Bill**: $34 (approximately 27.4%)

Although the increase in usage and costs is clear, these figures don't identify the specific causes.

### Suggested Checks
1. **Air Leaks**: Check for any air leaks in your home. Air leaks can lead to higher heating and cooling costs as they make it harder to maintain 

In [50]:
answer = await run_agent("What are some low-cost ways to reduce heating energy use?")
print(answer)


TOOL CALL: search_energy_guidance('What are some low-cost ways to reduce heating energy use?')
TOOL RESULT: search_energy_guidance returned 5 records
Here are some low-cost ways to reduce heating energy use based on information from the U.S. Department of Energy:

1. **Use a Programmable Thermostat**: Setting back your thermostat by 7°F to 10°F for 8 hours a day can result in savings of up to 10% on heating costs. Programmable thermostats allow for automatic adjustments, ensuring savings even when you're not home.

2. **Seal Air Leaks**: Use caulk or weather stripping to seal gaps around windows and doors. This can prevent warm air from escaping during winter, making your heating system more efficient.

3. **Change Ceiling Fan Direction**: In colder months, set ceiling fans to run in a clockwise direction on a low speed. This helps push warm air down from the ceiling, potentially saving up to 15% on your winter energy bills.

4. **Switch to Energy-Efficient Lighting**: If you’re using 

In [51]:
answer = await run_agent("Can you tell me something about capybaras?")
print(answer)


I’m the Home Energy Helper and can answer questions about household energy use, electricity bills, and ways to save energy. Is your question about one of those?
